<a href="https://colab.research.google.com/github/Prerna-Karle/genai-workshop/blob/main/AutonomousAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Full Agent Pipeline
!pip install -q transformers sentence-transformers ipywidgets wikipedia numpy scikit-learn

import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import sqlite3
import wikipedia
import numpy as np
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Booting up Agent Architecture...")
print("1/3 Loading Controller (Zero-Shot LLM)...")
controller_llm = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("2/3 Loading Memory (Vector Embedder)...")
memory_embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("3/3 Initializing Tools (SQLite Database)...")

# ==========================================
# TOOL 1: IN-MEMORY SQLITE DATABASE
# ==========================================
conn = sqlite3.connect(':memory:')
c = conn.cursor()
c.execute('''CREATE TABLE clients (company_name text, industry text, revenue real)''')
c.execute("INSERT INTO clients VALUES ('SpaceX', 'Aerospace', 8000000)")
c.execute("INSERT INTO clients VALUES ('Tesla', 'Automotive', 5000000)")
c.execute("INSERT INTO clients VALUES ('OpenAI', 'Technology', 2000000)")
conn.commit()

def tool_database_query(query_intent):
    """Simulates text-to-SQL for the agent."""
    if "highest revenue" in query_intent.lower() or "top client" in query_intent.lower():
        c.execute("SELECT company_name FROM clients ORDER BY revenue DESC LIMIT 1")
        result = c.fetchone()[0]
        return f"Database Result: {result} is the highest revenue client."
    return "Database Result: Error - Unknown query format."

# ==========================================
# TOOL 2: WIKIPEDIA WEB SEARCH
# ==========================================
def tool_wikipedia(query):
    try:
        search_res = wikipedia.search(query)
        if search_res:
            summary = wikipedia.summary(search_res[0], sentences=1)
            return f"Web Result: {summary}"
        return "Web Result: No information found."
    except Exception as e:
        return "Web Result: Error - Ambiguous search term."

# ==========================================
# LONG-TERM MEMORY (Vector DB)
# ==========================================
# The agent stores past facts. We embed them into math vectors.
past_facts = [
    "The user prefers outputs to be concise.",
    "Our previous top client was Tesla, but it changed this year.",
    "If Wikipedia fails, reflect and try a broader search."
]
memory_vectors = memory_embedder.encode(past_facts)

# ==========================================
# THE ADVANCED AGENT CLASS
# ==========================================
class AutonomousAgent:
    def __init__(self):
        self.short_term_memory = [] # Tracks the current loop
        self.plan_queue = []        # The Planner's step-by-step queue
        self.tools = ["Query Database", "Search Web", "Mathematical Calculation"]

    def planning_engine(self, goal):
        print("\n🧩 [PLANNING ENGINE]: Breaking down the complex goal into steps...")
        # A simple simulated planner: splits multi-part goals by "and then"
        if "and then" in goal:
            steps = goal.split("and then")
            self.plan_queue = [s.strip() for s in steps]
        else:
            self.plan_queue = [goal]

        for i, step in enumerate(self.plan_queue):
            print(f"   ↳ Step {i+1}: {step}")
        time.sleep(1)

    def retrieve_long_term_memory(self, current_step):
        print(f"\n🧠 [LONG-TERM MEMORY]: Searching past experiences for context related to: '{current_step}'")
        step_vec = memory_embedder.encode([current_step])
        scores = cosine_similarity(step_vec, memory_vectors)[0]
        best_match = np.argmax(scores)
        if scores[best_match] > 0.15: # Threshold for relevance
            print(f"   ↳ Retrieved Fact: \"{past_facts[best_match]}\" (Relevance: {scores[best_match]*100:.0f}%)")
        else:
            print("   ↳ No highly relevant past memories found.")
        time.sleep(1)

    def controller(self, step):
        print(f"\n⚙️ [CONTROLLER (LLM)]: Processing Step: '{step}'")
        print("   ↳ Deciding which tool to trigger based on step intent...")

        # The LLM scores the step against available tools
        decision = controller_llm(step, candidate_labels=self.tools)
        chosen_tool = decision['labels'][0]

        print(f"   ↳ Decision: Invoking [{chosen_tool}]")
        time.sleep(1)
        return chosen_tool

    def execute_tool(self, tool, step):
        print(f"\n🛠️ [ACTION/TOOLS]: Executing tool...")
        if tool == "Query Database":
            result = tool_database_query(step)
        elif tool == "Search Web":
            # If the step relies on previous info (like "that company"), resolve it from Short-Term Memory
            if "that company" in step.lower() or "it" in step.lower():
                print("   ↳ [REFLECTION]: Resolving pronoun 'that company' using Short-Term Memory...")
                # Extract the company name from the previous step's result
                previous_data = self.short_term_memory[-1]
                if "SpaceX" in previous_data:
                    step = step.replace("that company", "SpaceX")
                    print(f"   ↳ Adjusted Search Query: '{step}'")
            result = tool_wikipedia(step)
        else:
            result = "Result: No tool execution required."

        print(f"   ↳ Observation: {result}")
        time.sleep(1)
        return result

    def run(self, complex_goal):
        print("="*70)
        print(f"🎯 NEW MISSION: {complex_goal}")
        print("="*70)

        # 1. Plan
        self.planning_engine(complex_goal)

        step_counter = 1
        # 2. Execution Loop
        while self.plan_queue:
            print(f"\n--- EXECUTING SUB-TASK {step_counter} ---")
            current_step = self.plan_queue.pop(0)

            # 3. Retrieve Context
            self.retrieve_long_term_memory(current_step)

            # 4. Route via Controller
            tool = self.controller(current_step)

            # 5. Execute Action
            observation = self.execute_tool(tool, current_step)

            # 6. Update Short-Term Memory
            print(f"\n📝 [SHORT-TERM MEMORY]: Saving step result to conversational context...")
            self.short_term_memory.append(observation)

            # 7. Reflection / Error Handling
            if "Error" in observation:
                print(f"🚨 [PLANNING ENGINE REFLECTION]: Step failed! Adjusting plan and retrying...")
                self.plan_queue.insert(0, f"Retry: {current_step} with broader search terms")

            step_counter += 1

        print("\n" + "="*70)
        print("✅ [FINAL SYNTHESIS]: Mission Complete.")
        print(f"Summary of findings: Based on the database, {self.short_term_memory[0].replace('Database Result: ', '')} "
              f"Furthermore, web research confirms: {self.short_term_memory[1].replace('Web Result: ', '')}")
        print("="*70)

# ==========================================
# INTERACTIVE UI
# ==========================================
agent = AutonomousAgent()

user_input = widgets.Textarea(
    value="Find our highest revenue client in the database, and then search Wikipedia to find out who founded that company.",
    description='Objective:',
    layout=widgets.Layout(width='90%', height='60px')
)
run_btn = widgets.Button(description="Execute Autonomous Agent", button_style='success', layout=widgets.Layout(width='30%'))
output_area = widgets.Output()

def on_run(b):
    with output_area:
        clear_output()
        agent.run(user_input.value)

        # Newbie Student Explanation
        print("\n\n" + "*"*70)
        print("👨‍🏫 PROFESSOR'S EXPLANATION (How this works under the hood):")
        print("1. PLANNING ENGINE: Notice how the agent didn't just rush in. It read the 'and then' in your prompt and split the job into a queue of two distinct sub-tasks.")
        print("2. LONG-TERM MEMORY: Before attempting Step 1, it converted your text into math (vectors) to search its database of past rules, recalling that 'Tesla' used to be the top client.")
        print("3. CONTROLLER: The LLM acted as a traffic cop. It looked at Step 1, realized it was a data query, and routed it to the 'Query Database' API instead of Google.")
        print("4. SHORT-TERM MEMORY & REFLECTION: In Step 2, you asked about 'that company'. AI models have no memory by default. The agent had to look at its Short-Term Memory, read the result from Step 1 ('SpaceX'), and dynamically rewrite its own search query to 'SpaceX' before hitting Wikipedia. This is what makes it an 'Agent'!")
        print("*"*70)

run_btn.on_click(on_run)
display(user_input, run_btn, output_area)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 15.7 MB/s eta 0:00:00
Booting up Agent Architecture...
1/3 Loading Controller (Zero-Shot LLM)...


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

3/3 Initializing Tools (SQLite Database)...


Textarea(value='Find our highest revenue client in the database, and then search Wikipedia to find out who fou…

Button(button_style='success', description='Execute Autonomous Agent', layout=Layout(width='30%'), style=Butto…

Output()

In [ ]:
#Full Agent Pipeline
!pip install -q transformers sentence-transformers ipywidgets wikipedia numpy scikit-learn

import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import sqlite3
import wikipedia
import numpy as np
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Booting up Agent Architecture...")
print("1/3 Loading Controller (Zero-Shot LLM)...")
controller_llm = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("2/3 Loading Memory (Vector Embedder)...")
memory_embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("3/3 Initializing Tools (SQLite Database)...")

# ==========================================
# TOOL 1: IN-MEMORY SQLITE DATABASE
# ==========================================
conn = sqlite3.connect(':memory:')
c = conn.cursor()
c.execute('''CREATE TABLE clients (company_name text, industry text, revenue real)''')
c.execute("INSERT INTO clients VALUES ('SpaceX', 'Aerospace', 8000000)")
c.execute("INSERT INTO clients VALUES ('Tesla', 'Automotive', 5000000)")
c.execute("INSERT INTO clients VALUES ('OpenAI', 'Technology', 2000000)")
conn.commit()

def tool_database_query(query_intent):
    """Simulates text-to-SQL for the agent."""
    if "highest revenue" in query_intent.lower() or "top client" in query_intent.lower():
        c.execute("SELECT company_name FROM clients ORDER BY revenue DESC LIMIT 1")
        result = c.fetchone()[0]
        return f"Database Result: {result} is the highest revenue client."
    elif "lowest revenue" in query_intent.lower() or "modest client" in query_intent.lower():
        c.execute("SELECT company_name FROM clients ORDER BY revenue ASC LIMIT 1")
        result = c.fetchone()[0]
        return f"Database Result: {result} is the highest revenue client."
    return "Database Result: Error - Unknown query format."

# ==========================================
# TOOL 2: WIKIPEDIA WEB SEARCH
# ==========================================
def tool_wikipedia(query):
    try:
        search_res = wikipedia.search(query)
        if search_res:
            summary = wikipedia.summary(search_res[0], sentences=1)
            return f"Web Result: {summary}"
        return "Web Result: No information found."
    except Exception as e:
        return "Web Result: Error - Ambiguous search term."

# ==========================================
# LONG-TERM MEMORY (Vector DB)
# ==========================================
# The agent stores past facts. We embed them into math vectors.
past_facts = [
    "The user prefers outputs to be concise.",
    "Our previous top client was Tesla, but it changed this year.",
    "If Wikipedia fails, reflect and try a broader search."
]
memory_vectors = memory_embedder.encode(past_facts)

# ==========================================
# THE ADVANCED AGENT CLASS
# ==========================================
class AutonomousAgent:
    def __init__(self):
        self.short_term_memory = [] # Tracks the current loop
        self.plan_queue = []        # The Planner's step-by-step queue
        self.tools = ["Query Database", "Search Web", "Mathematical Calculation"]

    def planning_engine(self, goal):
        print("\n🧩 [PLANNING ENGINE]: Breaking down the complex goal into steps...")
        # A simple simulated planner: splits multi-part goals by "and then"
        if "and then" in goal:
            steps = goal.split("and then")
            self.plan_queue = [s.strip() for s in steps]
        else:
            self.plan_queue = [goal]

        for i, step in enumerate(self.plan_queue):
            print(f"   ↳ Step {i+1}: {step}")
        time.sleep(1)

    def retrieve_long_term_memory(self, current_step):
        print(f"\n🧠 [LONG-TERM MEMORY]: Searching past experiences for context related to: '{current_step}'")
        step_vec = memory_embedder.encode([current_step])
        scores = cosine_similarity(step_vec, memory_vectors)[0]
        best_match = np.argmax(scores)
        if scores[best_match] > 0.15: # Threshold for relevance
            print(f"   ↳ Retrieved Fact: \"{past_facts[best_match]}\" (Relevance: {scores[best_match]*100:.0f}%)")
        else:
            print("   ↳ No highly relevant past memories found.")
        time.sleep(1)

    def controller(self, step):
        print(f"\n⚙️ [CONTROLLER (LLM)]: Processing Step: '{step}'")
        print("   ↳ Deciding which tool to trigger based on step intent...")

        # The LLM scores the step against available tools
        decision = controller_llm(step, candidate_labels=self.tools)
        chosen_tool = decision['labels'][0]

        print(f"   ↳ Decision: Invoking [{chosen_tool}]")
        time.sleep(1)
        return chosen_tool

    def execute_tool(self, tool, step):
        print(f"\n🛠️ [ACTION/TOOLS]: Executing tool...")
        if tool == "Query Database":
            result = tool_database_query(step)
        elif tool == "Search Web":
            # If the step relies on previous info (like "that company"), resolve it from Short-Term Memory
            if "that company" in step.lower() or "it" in step.lower():
                print("   ↳ [REFLECTION]: Resolving pronoun 'that company' using Short-Term Memory...")
                # Extract the company name from the previous step's result
                previous_data = self.short_term_memory[-1]
                if "SpaceX" in previous_data:
                    step = step.replace("that company", "SpaceX")
                    print(f"   ↳ Adjusted Search Query: '{step}'")
            result = tool_wikipedia(step)
        else:
            result = "Result: No tool execution required."

        print(f"   ↳ Observation: {result}")
        time.sleep(1)
        return result

    def run(self, complex_goal):
        print("="*70)
        print(f"🎯 NEW MISSION: {complex_goal}")
        print("="*70)

        # 1. Plan
        self.planning_engine(complex_goal)

        step_counter = 1
        # 2. Execution Loop
        while self.plan_queue:
            print(f"\n--- EXECUTING SUB-TASK {step_counter} ---")
            current_step = self.plan_queue.pop(0)

            # 3. Retrieve Context
            self.retrieve_long_term_memory(current_step)

            # 4. Route via Controller
            tool = self.controller(current_step)

            # 5. Execute Action
            observation = self.execute_tool(tool, current_step)

            # 6. Update Short-Term Memory
            print(f"\n📝 [SHORT-TERM MEMORY]: Saving step result to conversational context...")
            self.short_term_memory.append(observation)

            # 7. Reflection / Error Handling
            if "Error" in observation:
                print(f"🚨 [PLANNING ENGINE REFLECTION]: Step failed! Adjusting plan and retrying...")
                self.plan_queue.insert(0, f"Retry: {current_step} with broader search terms")

            step_counter += 1

        print("\n" + "="*70)
        print("✅ [FINAL SYNTHESIS]: Mission Complete.")
        print(f"Summary of findings: Based on the database, {self.short_term_memory[0].replace('Database Result: ', '')} "
              f"Furthermore, web research confirms: {self.short_term_memory[1].replace('Web Result: ', '')}")
        print("="*70)

# ==========================================
# INTERACTIVE UI
# ==========================================
agent = AutonomousAgent()

user_input = widgets.Textarea(
    value="Find our highest revenue client in the database, and then search Wikipedia to find out who founded that company.",
    description='Objective:',
    layout=widgets.Layout(width='90%', height='60px')
)
run_btn = widgets.Button(description="Execute Autonomous Agent", button_style='success', layout=widgets.Layout(width='30%'))
output_area = widgets.Output()

def on_run(b):
    with output_area:
        clear_output()
        agent.run(user_input.value)

        # Newbie Student Explanation
        print("\n\n" + "*"*70)
        print("👨‍🏫 PROFESSOR'S EXPLANATION (How this works under the hood):")
        print("1. PLANNING ENGINE: Notice how the agent didn't just rush in. It read the 'and then' in your prompt and split the job into a queue of two distinct sub-tasks.")
        print("2. LONG-TERM MEMORY: Before attempting Step 1, it converted your text into math (vectors) to search its database of past rules, recalling that 'Tesla' used to be the top client.")
        print("3. CONTROLLER: The LLM acted as a traffic cop. It looked at Step 1, realized it was a data query, and routed it to the 'Query Database' API instead of Google.")
        print("4. SHORT-TERM MEMORY & REFLECTION: In Step 2, you asked about 'that company'. AI models have no memory by default. The agent had to look at its Short-Term Memory, read the result from Step 1 ('SpaceX'), and dynamically rewrite its own search query to 'SpaceX' before hitting Wikipedia. This is what makes it an 'Agent'!")
        print("*"*70)

run_btn.on_click(on_run)
display(user_input, run_btn, output_area)